# G1: コロナショック耐性モデル

**目的**: 「コロナ禍（2020→2021）の変化率」を目的変数にした LightGBM モデルを構築し、
SHAP で「コロナショックを受けにくい路線の条件」を明らかにする。

| | Phase B（既存） | G1（今回） |
|---|---|---|
| 目的変数 | log(路線価) ← 価格の予測 | chg_20_21 ← 変化率の予測 |
| 問い | 来年の路線価はいくら？ | **コロナで下落しやすい条件は何か？** |
| SHAP の意味 | 価格を上げる/下げる特徴 | **耐性を高める/下げる特徴** |

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import japanize_matplotlib
except ImportError:
    pass

import lightgbm as lgb
import shap
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.spatial import cKDTree

from compare_years import build_panel

plt.rcParams["figure.dpi"] = 120

PATH_2022 = Path("/Users/KASU/REX_data_2020-2022/2022/nouhin_line_2022.shp")
PATH_2020 = Path("/Users/KASU/REX_data_2020-2022/2020/nouhin_line_2020.shp")
OUT_DIR   = Path("../outputs")

RED  = "#E8534A"
BLUE = "#3A7FC1"
NAVY = "#1B2B4B"
GOLD = "#D4A843"

print("セットアップ完了")

## 1. パネルデータ読み込み・特徴量生成

In [ ]:

panel = build_panel(PATH_2022, PATH_2020)

# build_panel のカラム名確認
# kakaku → price_2022, pre1_kakak → price_2021, pre2_kakak → price_2020
TARGET = "chg_20_21"

req_cols = [TARGET, "chg_18_19", "chg_19_20", "price_2022", "chikukbn", "pref_code"]
df = panel.dropna(subset=req_cols).copy()
df = df[df["price_2022"] > 0]
print(f"有効路線数: {len(df):,}")
print(f"目的変数 {TARGET} の記述統計:")
print(df[TARGET].describe().round(3))


In [ ]:

# ── 特徴量エンジニアリング ─────────────────────────────────────────────

df_proj = df.to_crs(epsg=3857).copy()
centroids = np.column_stack([
    df_proj.geometry.centroid.x.values,
    df_proj.geometry.centroid.y.values,
])

# 1) 都市中心からの距離（EPSG:3857 座標）
CITIES_3857 = {
    "tokyo":   (15554819, 4252810),
    "osaka":   (15220000, 4143000),
    "fukuoka": (14411000, 3946000),
    "sapporo": (15715000, 5081000),
    "sendai":  (15799000, 4637000),
}
for city, (cx, cy) in CITIES_3857.items():
    dist = np.sqrt((centroids[:, 0] - cx)**2 + (centroids[:, 1] - cy)**2) / 1000
    df[f"dist_{city}_km"] = dist.astype(np.float32)

# 2) 重心座標（lat/lon）
centroids_4326 = df.geometry.centroid
df["centroid_lon"] = centroids_4326.x.astype(np.float32)
df["centroid_lat"] = centroids_4326.y.astype(np.float32)

# 3) 地区区分 One-hot
for c in range(1, 8):
    df[f"chiku_{c}"] = (df["chikukbn"] == c).astype(np.int8)
df["is_commercial"] = df["chikukbn"].isin([1, 2, 3, 4]).astype(np.int8)

# 4) 価格水準（対数）― build_panel のカラム名を使用
df["log_kakaku"] = np.log1p(df["price_2022"]).astype(np.float32)
df["log_pre1"]   = np.log1p(df["price_2021"]).astype(np.float32)

# 5) コロナ前の変化率モメンタム（外れ値クリップ）
df["chg_18_19_clip"] = df["chg_18_19"].clip(-30, 30).astype(np.float32)
df["chg_19_20_clip"] = df["chg_19_20"].clip(-30, 30).astype(np.float32)

# 6) 近傍8路線の空間ラグ（コロナ前変化率・価格水準）
#    ※ 目的変数(chg_20_21)の空間ラグは使わない（リーク防止）
tree = cKDTree(centroids)
_, nn_idx = tree.query(centroids, k=9)  # 自身 + 8近傍
nn_idx = nn_idx[:, 1:]  # 自身除く

pre_chg_vals  = df["chg_18_19_clip"].values
log_kaku_vals = df["log_kakaku"].values
df["lag8_pre_chg_mean"]    = pre_chg_vals[nn_idx].mean(axis=1).astype(np.float32)
df["lag8_log_kakaku_mean"] = log_kaku_vals[nn_idx].mean(axis=1).astype(np.float32)

print("特徴量生成完了")
print(f"  特徴量例: {[c for c in df.columns if c.startswith('dist_') or c.startswith('lag')]}")


## 2. LightGBM 学習（Spatial CV）

In [ ]:
FEATURE_COLS = [
    # コロナ前の変化率（モメンタム）
    "chg_18_19_clip", "chg_19_20_clip",
    # 近傍のコロナ前変化率（空間ラグ）
    "lag8_pre_chg_mean", "lag8_log_kakaku_mean",
    # 価格水準
    "log_kakaku", "log_pre1",
    # 地区区分
    "chiku_1", "chiku_2", "chiku_3", "chiku_4",
    "chiku_5", "chiku_6", "chiku_7", "is_commercial",
    # 都市距離
    "dist_tokyo_km", "dist_osaka_km", "dist_fukuoka_km",
    "dist_sapporo_km", "dist_sendai_km",
    # 位置
    "centroid_lat", "centroid_lon",
]

X = df[FEATURE_COLS].astype(np.float32)
y = df[TARGET].clip(-30, 30).astype(np.float32)  # 外れ値クリップ
groups = df["pref_code"]

params = {
    "objective":         "regression",
    "metric":            "rmse",
    "n_estimators":      500,
    "learning_rate":     0.05,
    "num_leaves":        63,
    "min_child_samples": 50,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "random_state":      42,
    "n_jobs":            -1,
    "verbose":           -1,
}

gkf = GroupKFold(n_splits=5)
fold_results = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    fold_results.append({
        "fold": fold + 1,
        "n_test": len(te_idx),
        "rmse": round(np.sqrt(mean_squared_error(y_te, y_pred)), 4),
        "mae":  round(mean_absolute_error(y_te, y_pred), 4),
        "r2":   round(r2_score(y_te, y_pred), 4),
    })
    print(f"  Fold {fold+1}: RMSE={fold_results[-1]['rmse']:.4f}, R²={fold_results[-1]['r2']:.4f}")

cv_df = pd.DataFrame(fold_results)
print("\nCV 平均:")
print(cv_df[['rmse','mae','r2']].mean().round(4))

# フル学習（SHAP用）
model_full = lgb.LGBMRegressor(**params)
model_full.fit(X, y)
print("\nフルモデル学習完了")

## 3. SHAP 分析

In [ ]:
# SHAP 計算（サンプル5万件）
SHAP_N = 50_000
X_sample = X.sample(SHAP_N, random_state=42)

explainer   = shap.TreeExplainer(model_full)
shap_values = explainer.shap_values(X_sample)

print(f"SHAP 計算完了: shape={shap_values.shape}")

In [ ]:
# ── SHAP Beeswarm（スライド用） ─────────────────────────────────────────
# 特徴量名を日本語に変換
FEAT_LABELS = {
    "chg_18_19_clip":      "コロナ前変化率①（2018→19）",
    "chg_19_20_clip":      "コロナ前変化率②（2019→20）",
    "lag8_pre_chg_mean":   "近傍8路線のコロナ前変化率（空間ラグ）",
    "lag8_log_kakaku_mean":"近傍8路線の価格水準（空間ラグ）",
    "log_kakaku":          "価格水準（対数）",
    "log_pre1":            "前年価格水準（対数）",
    "is_commercial":       "商業地フラグ",
    "chiku_1":             "地区区分: 高度商業",
    "chiku_2":             "地区区分: 繁華街",
    "chiku_3":             "地区区分: 普通商業",
    "chiku_4":             "地区区分: 併用住宅",
    "chiku_5":             "地区区分: 普通住宅",
    "chiku_6":             "地区区分: 中小工場",
    "chiku_7":             "地区区分: 大工場",
    "dist_tokyo_km":       "東京からの距離（km）",
    "dist_osaka_km":       "大阪からの距離（km）",
    "dist_fukuoka_km":     "福岡からの距離（km）",
    "dist_sapporo_km":     "札幌からの距離（km）",
    "dist_sendai_km":      "仙台からの距離（km）",
    "centroid_lat":        "緯度",
    "centroid_lon":        "経度",
}

X_labeled = X_sample.rename(columns=FEAT_LABELS)
shap_df = pd.DataFrame(shap_values, columns=X_sample.columns)

# 平均|SHAP|で上位15特徴量に絞る
mean_abs = shap_df.abs().mean().sort_values(ascending=False)
top15 = mean_abs.head(15).index.tolist()
top15_labels = [FEAT_LABELS.get(c, c) for c in top15]

fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values[:, [X_sample.columns.get_loc(c) for c in top15]],
    X_sample[top15],
    feature_names=top15_labels,
    plot_type="dot",
    max_display=15,
    show=False,
    color_bar=True,
)
ax = plt.gca()
ax.set_xlabel("SHAP 値（正 = コロナ禍でも上昇しやすい　負 = 下落しやすい）", fontsize=11)
ax.set_title(
    "コロナショック耐性モデル — 特徴量の影響度（SHAP Beeswarm）\n"
    "特徴量値: 高（赤）→ 低（青）",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_covid_shock.png", dpi=150, bbox_inches="tight")
plt.show()
print("保存: shap_covid_shock.png")

In [ ]:
# ── SHAP 平均重要度バーチャート（スライド補助）─────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

top10 = mean_abs.head(10)
labels = [FEAT_LABELS.get(c, c) for c in top10.index]
colors = [RED if v > 0 else BLUE for v in top10.values]  # 全部正なのでREDに統一

ax.barh(range(len(top10)), top10.values[::-1], color=RED, alpha=0.8)
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(labels[::-1], fontsize=10)
ax.set_xlabel("平均 |SHAP値|（コロナ変化率への影響の大きさ）", fontsize=10)
ax.set_title("コロナ禍の変化率に最も影響した特徴量 TOP10", fontsize=12, fontweight="bold")
ax.axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig(OUT_DIR / "shap_importance_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("保存: shap_importance_bar.png")

In [ ]:
# ── 解釈サマリーを出力 ─────────────────────────────────────────────────
print("=== SHAP 解釈サマリー ===")
print("\n上位特徴量と耐性への影響（正=耐性あり, 負=脆弱）:")
for col in top15:
    label = FEAT_LABELS.get(col, col)
    shap_col = shap_df[col]
    corr = np.corrcoef(X_sample[col].fillna(0), shap_col)[0, 1]
    direction = "↑ 高いほど耐性あり" if corr > 0 else "↓ 高いほど脆弱"
    print(f"  {label}: {direction} (相関={corr:.2f})")

## 4. CV 結果・解釈まとめ

In [ ]:
print("=== Spatial CV 結果 ===")
print(cv_df.to_string(index=False))
print(f"\n平均 RMSE: {cv_df['rmse'].mean():.4f} pp")
print(f"平均 R²  : {cv_df['r2'].mean():.4f}")
print()
print("※ R² が低くても問題ない: 変化率は価格より本質的に予測困難")
print("  重要なのは『どの特徴量が耐性に効くか』のSHAP解釈")

# 結果をCSV保存
cv_df.to_csv(OUT_DIR / "cv_results_shock.csv", index=False)
print("\n保存: cv_results_shock.csv")